In [1]:
import pandas as pd
import numpy as np
import requests

In [12]:
API_KEY = 'SUhElnAP0acVBL4OAUN5hvEmg8YwJMbd' 
SYMBOL = 'SOXL'

today = pd.Timestamp.today()
mid_data = (today-pd.DateOffset(months=6)).strftime('%Y-%m-%d')
start_date  = (today - pd.DateOffset(years=1)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')

print(f"Downloading data for {SYMBOL} from {start_date} to {mid_data}... (First 6 months)")
url_1 = f"https://api.polygon.io/v2/aggs/ticker/{SYMBOL}/range/5/minute/{start_date}/{mid_data}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}"
response_1 = requests.get(url_1)
data_1 = response_1.json()

print(f"Downloading data for {SYMBOL} from {mid_data} to {end_date}... (Another 6 months)")
url_2 = f"https://api.polygon.io/v2/aggs/ticker/{SYMBOL}/range/5/minute/{mid_data}/{end_date}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}"
response_2 = requests.get(url_2)
data_2 = response_2.json()

# 두 구간 데이터 합치기
raw_data = data_1.get('results', []) + data_2.get('results', [])

# results 가 data에 존재. 그 안에 내용물이 1개이상 존재하는지.
if len(raw_data) > 0:
    df = pd.DataFrame(raw_data)
# 원본데이터의 복잡한 time을 알기쉽게 변한(밀리초임) utc시간대로 변환.
    df['time'] = pd.to_datetime(df['t'], unit='ms', utc=True)
    # 방금만든 time을 인덱스로 설정. inplace=True는 원본데이터를 바로 바꾸겠다는 의미.
    df.set_index('time', inplace=True)

    df.index = df.index.tz_convert('America/Vancouver')
    
    full_df = df.rename(columns={'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume'})
    full_df.sort_index(inplace=True)

    full_df = full_df[~full_df.index.duplicated(keep='first')]
    full_df = full_df.between_time('06:35', '12:55')

    print(f"{len(full_df)} rows of data downloaded.")
    print("Data start time:", full_df.index[0])
    print("Data end time:", full_df.index[-1])

else:
    print("No data received from the Polygon API.")



8050 rows of data downloaded.
Data start time: 2025-09-29 06:35:00-07:00
Data end time: 2026-06-11 12:55:00-07:00


In [17]:
# buy_trades = full_df.at_time('06:35')
# sell_trades = full_df.at_time('12:55')

# buy_df = pd.DataFrame({'Buy_Price': buy_trades['open']}, index=buy_trades.index.date)
# sell_df = pd.DataFrame({'Sell_Price': sell_trades['close']}, index=sell_trades.index.date)

# # 날짜 중복 방지 (첫번째 거래 데이터 유지)
# buy_df = buy_df[~buy_df.index.duplicated(keep='first')]
# sell_df = sell_df[~sell_df.index.duplicated(keep='first')]

# backtest_df = pd.concat([buy_df, sell_df], axis=1).dropna()

# backtest_df['Return_%'] = (backtest_df['Sell_Price'] - backtest_df['Buy_Price']) / backtest_df['Buy_Price'] * 100
# backtest_df['Cumulative_Return_%'] = ((1 + backtest_df['Return_%'] / 100).cumprod() - 1) *100 


# display(backtest_df.head(10))

# 1. 날짜(date) 컬럼 생성
full_df['date'] = full_df.index.date

# 2. 날짜별로 가장 첫 번째 봉(06:35 매수)과 마지막 봉(12:55 매도) 추출
daily_buy = full_df.groupby('date').first()
daily_sell = full_df.groupby('date').last()

# 3. 백테스트 DataFrame 구성 및 결측치 제거
backtest_df = pd.DataFrame({
    'Buy_Price': daily_buy['open'],
    'Sell_Price': daily_sell['close']
}).dropna()

# 4. 수익률 및 누적 수익률 계산
backtest_df['Return_%'] = (backtest_df['Sell_Price'] - backtest_df['Buy_Price']) / backtest_df['Buy_Price'] * 100
backtest_df['Cumulative_Return_%'] = ((1 + backtest_df['Return_%'] / 100).cumprod() - 1) * 100

# 5. 결과 확인
display(backtest_df.head(10))
display(backtest_df.tail(10))

,Buy_Price,Sell_Price,Return_%,Cumulative_Return_%
date,,,,
2025-09-29,35.230,34.0000,-3.491343,-3.491343
2025-09-30,34.020,34.8600,2.469136,-1.108413
2025-10-01,34.445,36.8700,7.040209,5.853762
2025-10-02,38.750,39.0399,0.748129,6.645684
2025-10-03,39.660,38.2300,-3.605648,2.800416
2025-10-06,42.600,41.6800,-2.159624,0.580313
2025-10-07,42.905,38.8450,-9.462767,-8.937367
2025-10-08,39.010,42.8250,9.779544,-0.031857
2025-10-09,42.580,42.2500,-0.775012,-0.806622


,Buy_Price,Sell_Price,Return_%,Cumulative_Return_%
date,,,,
2026-05-29,230.1300,224.0600,-2.637640,60.496963
2026-06-01,220.9850,226.9200,2.685703,64.807434
2026-06-02,245.3001,266.3700,8.589438,78.963467
2026-06-03,277.6050,280.0000,0.862737,80.507450
2026-06-04,248.4900,262.8200,5.766832,90.917011
2026-06-05,228.3300,182.1101,-20.242587,52.270468
2026-06-08,205.6400,211.1800,2.694028,56.372677
2026-06-09,220.5600,201.7813,-8.514100,43.058951
2026-06-10,190.4500,180.0250,-5.473878,35.228079
